# B2-022-probabilistic-latent-models — Practice p08 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** kl-divergence

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

SEED = 20260927
ATOL = 1e-12
RTOL = 1e-12
P1 = torch.tensor([0.5, 0.5, 0.0], dtype=torch.float64)
Q1 = torch.tensor([0.25, 0.25, 0.5], dtype=torch.float64)
P2 = torch.tensor([0.1, 0.0, 0.9], dtype=torch.float64)
Q2 = torch.tensor([0.3, 0.4, 0.3], dtype=torch.float64)
P3 = torch.tensor([0.4, 0.4, 0.2], dtype=torch.float64)
Q3 = torch.tensor([0.2, 0.4, 0.4], dtype=torch.float64)
P4 = torch.tensor([0.2, 0.3, 0.5], dtype=torch.float64)

## Solution

Vectorized with the Session 2 conventions:

- terms with $p_i=0$ are set to exactly $0$ by `torch.where` (no $0\cdot\log0$ evaluation leaks a `nan`);
- if any $p_i>0$ has $q_i=0$ the result is `+inf`, returned before any logarithm is taken;
- otherwise the sum of $p_i\log(p_i/q_i)$ over the support is finite.

In [ ]:
def discrete_kl(p, q):
    if p.dim() != 1 or q.dim() != 1:
        raise ValueError("p and q must be 1-D")
    if p.shape[0] != q.shape[0]:
        raise ValueError("p and q must have equal length")
    for name, v in (("p", p), ("q", q)):
        if bool((v < 0).any()):
            raise ValueError(f"{name} has a negative entry")
        if abs(v.sum().item() - 1.0) > 1e-9:
            raise ValueError(f"{name} does not sum to 1")
    support = p > 0
    if bool((support & (q == 0)).any()):
        return torch.tensor(float("inf"), dtype=torch.float64)
    safe_p = torch.where(support, p, torch.ones_like(p))
    safe_q = torch.where(support, q, torch.ones_like(q))
    terms = torch.where(support, p * torch.log(safe_p / safe_q), torch.zeros_like(p))
    return terms.sum().to(torch.float64)


returned = []
def kl(p, q):
    value = discrete_kl(p, q)
    returned.append(value)
    return value

kl_p1q1, kl_q1p1 = kl(P1, Q1), kl(Q1, P1)
kl_2 = kl(P2, Q2)
kl_3 = kl(P3, Q3)
kl_4 = kl(P4, P4)

g = torch.Generator().manual_seed(SEED)
U = torch.rand(200, 2, 5, generator=g, dtype=torch.float64)
pk = U[:, 0] / U[:, 0].sum(dim=1, keepdim=True)
qk = U[:, 1] / U[:, 1].sum(dim=1, keepdim=True)
forward = torch.stack([kl(pk[k], qk[k]) for k in range(200)])
reverse = torch.stack([kl(qk[k], pk[k]) for k in range(200)])
selfkl = torch.stack([kl(pk[k], pk[k]) for k in range(200)])

def raises_value_error(v):
    try:
        discrete_kl(v, torch.tensor([0.5, 0.5], dtype=torch.float64))
    except ValueError:
        return True
    return False

bad_sum = raises_value_error(torch.tensor([0.5, 0.6], dtype=torch.float64))
bad_negative = raises_value_error(torch.tensor([1.2, -0.2], dtype=torch.float64))
print(kl_p1q1.item(), kl_q1p1.item(), kl_2.item(), kl_3.item(), kl_4.item(), (forward - reverse).abs().max().item())

### Answer check

In [ ]:
import math
assert kl_p1q1.dim() == 0 and kl_p1q1.dtype == torch.float64
assert torch.allclose(kl_p1q1, torch.tensor(math.log(2), dtype=torch.float64), atol=ATOL, rtol=RTOL)
assert math.isinf(kl_q1p1.item()) and kl_q1p1.item() > 0
assert torch.allclose(kl_2, torch.tensor(0.8 * math.log(3), dtype=torch.float64), atol=ATOL, rtol=RTOL)
assert abs(kl_2.item() - 0.8788898309) < 1e-9
assert torch.allclose(kl_3, torch.tensor(0.2 * math.log(2), dtype=torch.float64), atol=ATOL, rtol=RTOL)
assert abs(kl_3.item() - 0.1386294361) < 1e-9
assert abs(kl_4.item()) <= ATOL
assert bool((forward >= 0).all())
assert bool((selfkl.abs() <= 1e-12).all())
assert bool(((forward - reverse).abs() > 1e-3).any())
assert bad_sum and bad_negative
assert not any(bool(torch.isnan(v)) for v in returned)